# GENIE systematics inspection (Product B — per-knob)

**Preferred input:** the PRL assembled per-knob file (all waves merged once):
`…/PRL/systematics/productB_sel_mup/GENIE/cov_mat_dict_per_knob.pkl`
plus `knob_mode_map.json` (wave + production mode + **Ar23p-distributed** mode).
Built by `scripts/build_prl_genie_per_knob.py` from Aug24 modes + Sep12 Ar23p +
VecFF + May EDepFSI FSI π/N only.

**Retired EDepFSI twins** (not in the PRL file): `EDepFSI_*MEC*`,
`EDepFSI_VecFFCCQEshape`, `EDepFSI_CoulombCCQE`. Keep **VecFFCCQEshape** via SBN_v1 VecFF.

Do **not** load the May combined pickle as the primary source — it mixes Old-era
knobs (MvA, SBNNuSyst SF/CRPA, …). Legacy path is now
`systematics-final-archive`.

**Sections**
1. **Per-mode knobs** — pick a mode (QE, MEC, …); show each knob’s contribution
2. **Per-mode totals** — mode contributions + total GENIE; frac. cov + corr per mode
3. **Top knobs + total** — total GENIE + top 10 *integrated* families; cov/corr for total
4. **(Archive) EDepFSI twin overlays** — diagnostic only; twins not in totals

Helpers: `analysis_village.numucc_1p0pi.syst_genie_inspect`.

**Note:** Aug24 mode disks only cover 4 vars (`integrated` + 3 TKIs). Ar23p / VecFF /
EDepFSI FSI cover all 23 Product B vars — so non-TKI plots are dominated by those
waves until Aug24 is reprocessed for full Product B.

In [ ]:
%load_ext autoreload
%autoreload 2


In [2]:
import os
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

REPO = Path("/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana")
sys.path.insert(0, str(REPO))

from analysis_village.numucc_1p0pi.dataset_locations import PLOTS_BASE
from analysis_village.numucc_1p0pi.syst_pipeline_walker import final_stage_var_configs
from analysis_village.numucc_1p0pi.syst_genie_inspect import (
    DISTRIBUTED_MODE_ORDER,
    EDEPFSI_TWIN_BASES,
    MODE_ORDER,
    PHYSICS_MODES,
    all_knob_parts,
    assign_ar23p_knob_to_mode,
    assign_knob_to_mode,
    assign_other_mode_knob_to_bucket,
    collect_edepfsi_twin_parts,
    contribution_rows,
    display_mode_name,
    integrated_frac_unc_pct,
    is_retired_edepfsi_twin,
    knob_parts_for_mode,
    load_combined_as_mode_groups,
    load_cov_mat_dict,
    load_groups,
    merge_cov_mat_dict_into_groups,
    mode_totals_ar23p_distributed,
    mode_totals_ar23p_standalone,
    plot_frac_unc_breakdown,
    plot_mode_frac_unc,
    show_cov_corr_heatmaps,
    sum_cov_fracs,
    top_n_knobs_by_integrated,
    write_contribution_csv,
)

# ---------------------------------------------------------------------------
# Configuration — New sources (same as systematics-genie-comparison)
# ---------------------------------------------------------------------------
SYST_BASE = Path("/exp/sbnd/data/users/munjung/xsec/numucc_1p0pi")
PLOTS = Path(PLOTS_BASE)

# Assembled per-knob Product B file (preferred). Rebuild with:
#   python analysis_village/numucc_1p0pi/scripts/build_prl_genie_per_knob.py
PRL_GENIE = Path(
    os.environ.get(
        "GENIE_INSPECT_PRL",
        str(SYST_BASE / "PRL/systematics/productB_sel_mup/GENIE"),
    )
)
PRL_PER_KNOB_PKL = PRL_GENIE / "cov_mat_dict_per_knob.pkl"
PRL_KNOB_MODE_MAP = PRL_GENIE / "knob_mode_map.json"
USE_PRL_PER_KNOB = os.environ.get("GENIE_INSPECT_USE_PRL", "1") not in (
    "0", "false", "False", ""
) and PRL_PER_KNOB_PKL.is_file()

_AR23P_NEW = [
    SYST_BASE / "syst_disk_sel_mup_20260912_Ar23p/GENIE/cov_mat_dict.pkl",
    SYST_BASE / "syst_disk_sel_mup_Aug24_Ar23p/GENIE/cov_mat_dict.pkl",
]
_ar23p_new = next((p for p in _AR23P_NEW if p.is_file()), _AR23P_NEW[0])

# Legacy wave-by-wave sources (fallback if PRL file missing / USE_PRL=0)
GROUP_SOURCES = {
    "CCQE": SYST_BASE / "syst_disk_sel_mup_Aug24_CCQE/GENIE/cov_mat_dict.pkl",
    "MEC": SYST_BASE / "syst_disk_sel_mup_Aug24_MEC/GENIE/cov_mat_dict.pkl",
    "RES": SYST_BASE / "syst_disk_sel_mup_Aug24_RES/GENIE/cov_mat_dict.pkl",
    "nonRES": SYST_BASE / "syst_disk_sel_mup_Aug24_nonRES/GENIE/cov_mat_dict.pkl",
    "DIS": SYST_BASE / "syst_disk_sel_mup_Aug24_DIS/GENIE/cov_mat_dict.pkl",
    "Other": SYST_BASE / "syst_disk_sel_mup_Aug24_Other/GENIE/cov_mat_dict.pkl",
    "Ar23p": _ar23p_new,
}
for mode in list(GROUP_SOURCES):
    env = os.environ.get(f"GENIE_INSPECT_{mode}")
    if env:
        GROUP_SOURCES[mode] = Path(env)

# EDepFSI / VecFF — only used in legacy fallback merge path
EDEPFSI_PKL = Path(
    os.environ.get(
        "GENIE_EDEPFSI_PKL",
        str(SYST_BASE / "GENIE/combined/productB/cov_mat_dict.pkl"),
    )
)
VECFF_PKL = Path(
    os.environ.get(
        "GENIE_VECFF_PKL",
        str(SYST_BASE / "syst_disk_sel_mup_VecFF/GENIE/cov_mat_dict.pkl"),
    )
)

OUT_DIR = Path(os.environ.get("GENIE_INSPECT_OUT", str(PLOTS / "genie_syst_inspect")))
OUT_DIR.mkdir(parents=True, exist_ok=True)

_FINAL_VCS = final_stage_var_configs()
VARS = [vc.var_save_name for vc in _FINAL_VCS if vc.var_save_name]
_vars_env = os.environ.get("GENIE_INSPECT_VARS", "").strip()
if _vars_env:
    VARS = [v.strip() for v in _vars_env.split(",") if v.strip()]

COV_TYPES = ["rate", "xsec"]
SAVE_FIGS = True
SHOW_PLOTS = os.environ.get("GENIE_INSPECT_SHOW", "0") not in ("0", "false", "False", "")
if not SHOW_PLOTS:
    matplotlib.use("Agg")
FIG_DPI = 140
TOP_N_KNOBS = 10
INSPECT_MODE = os.environ.get("GENIE_INSPECT_MODE", "CCQE")

vc_by = {vc.var_save_name: vc for vc in _FINAL_VCS}
print("OUT_DIR =", OUT_DIR)
print("INSPECT_MODE =", INSPECT_MODE)
print("SHOW_PLOTS =", SHOW_PLOTS)
print(f"VARS ({len(VARS)}):", ", ".join(VARS))
print("USE_PRL_PER_KNOB =", USE_PRL_PER_KNOB, "→", PRL_PER_KNOB_PKL)
print("PRL_KNOB_MODE_MAP exists =", PRL_KNOB_MODE_MAP.is_file())
if not USE_PRL_PER_KNOB:
    print("Legacy GROUP_SOURCES:")
    for m, p in GROUP_SOURCES.items():
        print(f"  {m:7s} exists={Path(p).is_file()}  {p}")
    print("EDEPFSI_PKL =", EDEPFSI_PKL, "exists=", EDEPFSI_PKL.is_file())
    print("VECFF_PKL   =", VECFF_PKL, "exists=", VECFF_PKL.is_file())

OUT_DIR = /exp/sbnd/data/users/munjung/plots/numucc1p0pi/genie_syst_inspect
INSPECT_MODE = CCQE
SHOW_PLOTS = False
VARS (23): integrated, muon-p, muon-dir_z, proton-p, proton-dir_z, tki-del_Tp, tki-del_Tp_x, tki-del_Tp_y, tki-del_p, tki-del_alpha, tki-del_phi, muon-dir_phi, proton-dir_phi, muon-dir_x, muon-dir_y, proton-dir_x, proton-dir_y, vertex_x, vertex_y, vertex_z, muon-end_x, muon-end_y, muon-end_z
New GROUP_SOURCES:
  CCQE    exists=True  /exp/sbnd/data/users/munjung/xsec/numucc_1p0pi/syst_disk_sel_mup_Aug24_CCQE/GENIE/cov_mat_dict.pkl
  MEC     exists=True  /exp/sbnd/data/users/munjung/xsec/numucc_1p0pi/syst_disk_sel_mup_Aug24_MEC/GENIE/cov_mat_dict.pkl
  RES     exists=True  /exp/sbnd/data/users/munjung/xsec/numucc_1p0pi/syst_disk_sel_mup_Aug24_RES/GENIE/cov_mat_dict.pkl
  nonRES  exists=True  /exp/sbnd/data/users/munjung/xsec/numucc_1p0pi/syst_disk_sel_mup_Aug24_nonRES/GENIE/cov_mat_dict.pkl
  DIS     exists=True  /exp/sbnd/data/users/munjung/xsec/numucc_1p0pi/syst_disk_sel_m

In [ ]:
import json

if USE_PRL_PER_KNOB:
    # Preferred: one assembled file (all waves + forgotten knobs + EDepFSI FSI).
    groups = load_combined_as_mode_groups(PRL_PER_KNOB_PKL)
    KNOB_MODE_MAP = None
    if PRL_KNOB_MODE_MAP.is_file():
        KNOB_MODE_MAP = json.loads(PRL_KNOB_MODE_MAP.read_text())
        print(
            f"knob_mode_map: n_knobs={KNOB_MODE_MAP.get('n_knobs')}  "
            f"distributed={ {k: len(v) for k, v in KNOB_MODE_MAP.get('by_distributed_mode', {}).items()} }"
        )
else:
    # Legacy: load Aug24 modes + Ar23p, then fold VecFF + EDepFSI FSI.
    _sources = {m: p for m, p in GROUP_SOURCES.items() if Path(p).is_file()}
    _missing = [m for m in GROUP_SOURCES if m not in _sources]
    if _missing:
        print("WARNING: missing New modes (skipped):", _missing)
    if not _sources:
        raise FileNotFoundError("No New GROUP_SOURCES files found")

    groups = load_groups(_sources)

    if VECFF_PKL.is_file():
        n = merge_cov_mat_dict_into_groups(groups, load_cov_mat_dict(VECFF_PKL))
        print(f"merged New VecFF product: {n} keys from {VECFF_PKL}")
    else:
        print("VecFF product not found — skip")

    if EDEPFSI_PKL.is_file():
        edep_full = load_cov_mat_dict(EDEPFSI_PKL)
        edep_only = {}
        n_edep_keys = n_retired = 0
        for slug, row in edep_full.items():
            keep = {}
            for k, v in row.items():
                if k in ("genie", "genie_rate") or "EDepFSI" not in str(k):
                    continue
                if is_retired_edepfsi_twin(k):
                    n_retired += 1
                    continue
                keep[k] = v
            if keep:
                edep_only[slug] = keep
                n_edep_keys += len(keep)
        n = merge_cov_mat_dict_into_groups(groups, edep_only)
        print(
            f"merged EDepFSI FSI from {EDEPFSI_PKL}: {n}/{n_edep_keys} keys "
            f"(retired twins skipped at filter: {n_retired}; "
            f"rest of merge skip = binning mismatch vs New)"
        )
    else:
        print("WARNING: EDepFSI source missing:", EDEPFSI_PKL)
    KNOB_MODE_MAP = None

# Sanity: no MvA; nominal NormCCMEC / VecFF present; retired EDepFSI twins absent.
_sample = None
for mode in ("CCQE", "Ar23p", "MEC", "FSI"):
    if mode in groups and "integrated" in groups[mode]:
        _sample = groups[mode]["integrated"]
        break
if _sample is not None:
    mva = [k for k in _sample if "MvA" in k and not k.endswith("_rate")]
    print(f"MvA keys in sample mode (should be empty for New): {mva or 'none'}")
_all_int = {}
for g in groups.values():
    _all_int.update(g.get("integrated", {}))
print(
    "NormCCMEC (SBN_v1 nominal):",
    [k for k in _all_int if "NormCCMEC" in k and "EDepFSI" not in k and not k.endswith("_rate")]
    or "MISSING",
)
print(
    "NormCCMEC (EDepFSI, should be absent):",
    [k for k in _all_int if "EDepFSI_NormCCMEC" in k and not k.endswith("_rate")] or "none",
)
print(
    "VecFF (SBN_v1 nominal):",
    [k for k in _all_int if "VecFFCCQEshape" in k and "EDepFSI" not in k and not k.endswith("_rate")]
    or "MISSING",
)
print(
    "VecFF (EDepFSI, should be absent):",
    [k for k in _all_int if "EDepFSI_VecFFCCQEshape" in k and not k.endswith("_rate")] or "none",
)
print(
    "EDepFSI FSI π kept:",
    [k for k in _all_int if "EDepFSI" in k and "_pi" in k and not k.endswith("_rate")][:3]
    or "MISSING",
)

# Plot vars: any coverage (Aug24 is 4-var only).
_covered = set()
for cov in groups.values():
    _covered |= set(cov.keys())
VARS = [v for v in VARS if v in _covered]
if not VARS:
    raise RuntimeError("No overlapping variables between final_stage_var_configs and loaded groups")
print(f"plot VARS ({len(VARS)}):", ", ".join(VARS))
print("modes loaded:", sorted(groups.keys()))


## 1. Per-mode knob contributions

Set `INSPECT_MODE` in the config cell (or `GENIE_INSPECT_MODE`). For each analysis
variable, show **rate** and **xsec** fractional uncertainties from every knob in
that mode, plus the mode total.


In [ ]:
mode = INSPECT_MODE
if mode not in groups:
    raise KeyError(f"INSPECT_MODE={mode!r} not loaded; have {sorted(groups)}")

for slug in plot_vars:
    vc = vc_by.get(slug)
    for kind in ("xsec", "rate"):
        parts, total = knob_parts_for_mode(groups, mode, slug, kind)
        fig, ax = plt.subplots(figsize=(6.4, 4.8))
        plot_frac_unc_breakdown(
            ax, parts, total, vc=vc, kind=kind, top_n=None,
        )
        fig.tight_layout()
        if SAVE_FIGS:
            out = OUT_DIR / f"sec1_{mode}__{slug}__{kind}__knobs.png"
            fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
            print("wrote", out)
        if SHOW_PLOTS:
            plt.show()
        else:
            plt.close(fig)


## 2. Per-mode totals (+ Ar23p treatments)

For each variable × (rate, xsec):

* uncertainty curves for each mode and the GENIE total
* fractional covariance + correlation as one wide figure (subplots) **per mode**

**Two Ar23p treatments** (tag only appears in save names, not plot titles)

1. **Standalone** — Ar23p is its own mode  
2. **Distributed**
   - Ar23p: `QE`/`ZExp` → QE; `MEC` → MEC; else → **FSI**
   - Other mode: `*COH` / `*NCEL` → **Other**; remaining Other knobs → **FSI**


In [ ]:
def _section2(distribute_ar23p: bool, tag: str):
    mode_order = DISTRIBUTED_MODE_ORDER if distribute_ar23p else MODE_ORDER
    totals_fn = mode_totals_ar23p_distributed if distribute_ar23p else mode_totals_ar23p_standalone

    for kind in COV_TYPES:
        # --- mode uncertainty (one figure per variable) ---
        for slug in plot_vars:
            mt = totals_fn(groups, slug, kind)
            fig, ax = plt.subplots(figsize=(6.4, 4.8))
            plot_mode_frac_unc(
                ax, mt, vc=vc_by.get(slug), kind=kind, mode_order=mode_order,
            )
            fig.tight_layout()
            if SAVE_FIGS:
                out = OUT_DIR / f"sec2_mode_totals__{tag}__{kind}__{slug}.png"
                fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
                print("wrote", out)
            if SHOW_PLOTS:
                plt.show()
            else:
                plt.close(fig)

        # --- per-mode frac cov + corr (one wide figure) ---
        for slug in plot_vars:
            vc = vc_by.get(slug)
            if vc is None:
                print(f"skip heatmaps {slug}: no VariableConfig")
                continue
            mt = totals_fn(groups, slug, kind)
            for mode in mode_order:
                if mode not in mt:
                    continue
                save = None
                if SAVE_FIGS:
                    save = OUT_DIR / f"sec2_covcorr__{tag}__{kind}__{slug}__{mode}.png"
                show_cov_corr_heatmaps(
                    mt[mode], vc,
                    kind=kind,
                    suptitle=display_mode_name(mode),
                    save_path=save,
                    dpi=FIG_DPI,
                    show=SHOW_PLOTS,
                )

print("Distributed assignment examples:")
if "Other" in groups and "integrated" in groups["Other"]:
    from analysis_village.numucc_1p0pi.syst_genie_inspect import iter_knob_cov_fracs
    seen = {}
    for kn, _ in iter_knob_cov_fracs(groups["Other"]["integrated"], "xsec"):
        seen.setdefault(assign_other_mode_knob_to_bucket(kn), []).append(kn)
    for dest, kns in sorted(seen.items()):
        print(f"  Other→{dest}: {len(kns)} knobs (e.g. {kns[0]})")
if "Ar23p" in groups and "integrated" in groups["Ar23p"]:
    from analysis_village.numucc_1p0pi.syst_genie_inspect import iter_knob_cov_fracs
    seen = {}
    for kn, _ in iter_knob_cov_fracs(groups["Ar23p"]["integrated"], "xsec"):
        seen.setdefault(assign_ar23p_knob_to_mode(kn), []).append(kn)
    for dest, kns in sorted(seen.items()):
        print(f"  Ar23p→{display_mode_name(dest)}: {len(kns)} knobs (e.g. {kns[0]})")

_section2(distribute_ar23p=False, tag="standalone")
_section2(distribute_ar23p=True, tag="distributed")

## 3. Total GENIE + top 10 knobs

Ranking uses each knob’s fractional uncertainty on **`integrated`** (not an
average over bins of the plotted variable). Binned families (``ZExp_b*``, dials,
``q0bin*``, …) are collapsed into one knob each before ranking.

Plots show the total and those top 10 knobs for every analysis variable ×
(rate, xsec), plus frac. cov + corr for the **total**. Contribution percentages
are written to CSV next to the plots (not drawn on the legend). Legend shows
knob names only (no mode prefix).


In [9]:
for kind in COV_TYPES:
    # Rank on integrated (binned knobs collapsed into families)
    int_parts = all_knob_parts(
        groups, "integrated", kind, distribute_ar23p=False, collapse_binned=True,
    )
    top_keys = top_n_knobs_by_integrated(int_parts, int_parts, n=TOP_N_KNOBS)
    print(f"[{kind}] top {TOP_N_KNOBS} knobs by integrated frac. unc. (families):")
    for i, k in enumerate(top_keys, 1):
        print(f"  {i:2d}. {k:50s}  {integrated_frac_unc_pct(int_parts[k]):6.3f}%")

    for slug in plot_vars:
        vc = vc_by.get(slug)
        parts = all_knob_parts(
            groups, slug, kind, distribute_ar23p=False, collapse_binned=True,
        )
        mt = mode_totals_ar23p_standalone(groups, slug, kind)
        total = sum_cov_fracs(list(mt.values()))
        show_parts = {k: parts[k] for k in top_keys if k in parts}
        use_int_score = (slug == "integrated") or (vc is not None and getattr(vc, "var_save_name", "") == "integrated")

        fig, ax = plt.subplots(figsize=(6.4, 4.8))
        plot_frac_unc_breakdown(
            ax, show_parts, total, vc=vc,
            kind=kind,
            ordered_keys=top_keys,
            show_pct_in_legend=False,
            legend_knob_only=True,
        )
        fig.tight_layout()
        if SAVE_FIGS:
            out = OUT_DIR / f"sec3_top{TOP_N_KNOBS}__{kind}__{slug}.png"
            fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
            print("wrote", out)
            csv_rows = contribution_rows(
                show_parts, total, kind=kind, slug=slug,
                ordered_keys=top_keys, use_integrated_score=use_int_score,
            )
            csv_path = OUT_DIR / f"sec3_top{TOP_N_KNOBS}__{kind}__{slug}__contrib.csv"
            write_contribution_csv(csv_path, csv_rows)
            print("wrote", csv_path)
        if SHOW_PLOTS:
            plt.show()
        else:
            plt.close(fig)

        if total is not None and vc is not None:
            save = None
            if SAVE_FIGS:
                save = OUT_DIR / f"sec3_covcorr__{kind}__{slug}__total.png"
            show_cov_corr_heatmaps(
                total, vc, kind=kind, suptitle="GENIE",
                save_path=save, dpi=FIG_DPI, show=SHOW_PLOTS,
            )


[rate] top 10 knobs by integrated frac. unc. (families):
   1. Ar23p/ZExp                                          12.276%
   2. Ar23p/QEIntf                                        11.400%
   3. QE/RPA QE                                            8.171%
   4. Ar23p/LFGToSF                                        7.854%
   5. MEC/SBNNuSyst EDepFSI NormCCMEC                      7.058%
   6. MEC/NormCCMEC                                        7.045%
   7. Ar23p/LFGToHF                                        6.164%
   8. Ar23p/HFToCRPA                                       4.011%
   9. Ar23p/SuSAToVal MECResponse                          2.256%
  10. Ar23p/SuSAToMar MECResponse                          2.036%
wrote /exp/sbnd/data/users/munjung/plots/numucc1p0pi/genie_syst_inspect/sec3_top10__rate__integrated.png
wrote /exp/sbnd/data/users/munjung/plots/numucc1p0pi/genie_syst_inspect/sec3_top10__rate__integrated__contrib.csv
wrote /exp/sbnd/data/users/munjung/plots/numucc1p0pi/genie_syst_

In [ ]:
## 4. (Archive) EDepFSI twins vs nominal — not used in totals

Diagnostic overlays only. **Totals above use SBN_v1 nominals**, not these
`EDepFSI_*MEC*` twins (retired as duplicates).

- `NormCCMEC` (SBN_v1) vs `EDepFSI_NormCCMEC`
- `DecayAngMEC` (SBN_v1) vs `EDepFSI_DecayAngMEC`

Mats come from the May combined Product B MEC row (shared binning).

In [ ]:
# Twin pairs from May combined (nominal + EDepFSI share binning there).
_twin_groups = load_combined_as_mode_groups(EDEPFSI_PKL)
_twin_mec = {"MEC": _twin_groups["MEC"]} if "MEC" in _twin_groups else _twin_groups

TWIN_COLORS = {
    "NormCCMEC": "#1f77b4",
    "EDepFSI NormCCMEC": "#ff7f0e",
    "DecayAngMEC": "#2ca02c",
    "EDepFSI DecayAngMEC": "#d62728",
}

# Integrated summary: rate | xsec side-by-side for each pair.
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), sharey=False)
for ax, kind in zip(axes, COV_TYPES):
    labels, vals, colors = [], [], []
    for base in EDEPFSI_TWIN_BASES:
        parts = collect_edepfsi_twin_parts(_twin_mec, "integrated", kind, base)
        for lab in (base, f"EDepFSI {base}"):
            if lab not in parts:
                print(f"WARNING: missing integrated {kind} for {lab}")
                continue
            labels.append(lab)
            vals.append(integrated_frac_unc_pct(parts[lab]))
            colors.append(TWIN_COLORS.get(lab, "gray"))
    ax.barh(labels[::-1], vals[::-1], color=colors[::-1])
    ax.set_xlabel("Frac. unc. [%]")
    ax.set_title(f"integrated — {kind}")
    ax.grid(True, axis="x", alpha=0.3)
    for y, v in enumerate(vals[::-1]):
        ax.text(v, y, f" {v:.2f}%", va="center", fontsize=8)
fig.suptitle("EDepFSI twin vs nominal (integrated)", fontsize=12)
fig.tight_layout()
if SAVE_FIGS:
    out = OUT_DIR / "sec4_edepfsi_twins__integrated_summary.png"
    fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
    print("wrote", out)
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

# Per-variable overlays: one figure per (base, kind) with a small grid of vars.
_twin_slugs = [
    s for s in VARS
    if any(
        collect_edepfsi_twin_parts(_twin_mec, s, "rate", b)
        and collect_edepfsi_twin_parts(_twin_mec, s, "xsec", b)
        for b in EDEPFSI_TWIN_BASES
    )
]
print(f"twin plot vars ({len(_twin_slugs)}):", ", ".join(_twin_slugs))

for base in EDEPFSI_TWIN_BASES:
    for kind in COV_TYPES:
        usable = []
        for slug in _twin_slugs:
            parts = collect_edepfsi_twin_parts(_twin_mec, slug, kind, base)
            if set(parts) >= {base, f"EDepFSI {base}"}:
                if parts[base].shape == parts[f"EDepFSI {base}"].shape:
                    usable.append(slug)
        if not usable:
            print(f"skip {base}/{kind}: no vars with both sides")
            continue
        ncols = 3
        nrows = int(np.ceil(len(usable) / ncols))
        fig, axs = plt.subplots(
            nrows, ncols,
            figsize=(4.2 * ncols, 3.2 * nrows),
            squeeze=False,
        )
        ordered = [base, f"EDepFSI {base}"]
        for i, slug in enumerate(usable):
            r, c = divmod(i, ncols)
            ax = axs[r][c]
            parts = collect_edepfsi_twin_parts(_twin_mec, slug, kind, base)
            plot_frac_unc_breakdown(
                ax, parts, total=None,
                vc=vc_by.get(slug),
                kind=kind,
                ordered_keys=ordered,
                colors=TWIN_COLORS,
                show_pct_in_legend=True,
                show_title=True,
                title=slug,
            )
        for j in range(len(usable), nrows * ncols):
            r, c = divmod(j, ncols)
            axs[r][c].axis("off")
        fig.suptitle(f"{base}: nominal vs EDepFSI — {kind}", fontsize=13, y=1.01)
        fig.tight_layout()
        if SAVE_FIGS:
            out = OUT_DIR / f"sec4_edepfsi_twins__{base}__{kind}.png"
            fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
            print("wrote", out)
        if SHOW_PLOTS:
            plt.show()
        else:
            plt.close(fig)